## Simple CRAG - Corrective Retrieval Augmented Generation

In [ ]:
import os

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

In [ ]:
urls=[
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/"
]

In [ ]:
docs=[WebBaseLoader(url).load() for url in urls]
docs_list=[item for sublist in docs for item in sublist]

#from_tiktoken_encoder is a text splitter that counts chunk sizes and ideal for OpenAI model
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

doc_splits=text_splitter.split_documents(docs_list)

question="What is few-shot learning"

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
#add to a vector store
model_name = "sentence-transformers/all-MiniLM-L6-v2"
hf_embeddings = HuggingFaceEmbeddings(model_name=model_name)

vector_store=Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag_chroma",
    embedding=hf_embeddings
    #embedding=OpenAIEmbeddings() #or OpenAIEmbeddings(model="text-embedding-3-small") to specifiy a model
)

retriever=vector_store.as_retriever()

In [ ]:
docs=retriever.get_relevant_documents(question)

for doc in docs:
    print(doc.page_content[:50],"...",doc.metadata['source'])  #print out the first 50 characters along with the source metadata which in this case is the source url

## Retrieval Grader/The Corrective Part

In [ ]:
#Still in Langchain part not in LangGraph yet
from langchain_core.prompts import ChatPromptTemplate

#pydantic used for type hints and basically telling the user and system what is the expected output
#for specific inputs and outputs
from pydantic import BaseModel, Field 

from langchain_openai import ChatOpenAI

###### The ideal of a corrective RAG or CRAG framework is once we ask the system a question, it is going to grab documents and what we wanted it to do is to grade and correct that retrieval.  Can be considered as a re-ranking (re-ranking documents), but it will also remove things that are not relevant.

###### The data model below, called GradeDocuments, going to give a binary score, 'yes', or 'no', if this is a relevant document.  We will have an LLM read each one of those passages and answer the question if it's relevant

In [ ]:
#data model
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieval documents."""

    binary_score: str=Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )

In [ ]:
#llm to read each one of those passages
#temperature is used to control the randomness
llm=ChatOpenAI(model="gpt-4o-mini", temperature=0)

#with_structured_output returns a runnable which is a modified version of the original LLM
#design to enforce a structured output like GradeDocuments
structured_llm_grader=llm.with_structured_output(GradeDocuments)  #can force it a structure of type GradeDocuments

In [ ]:
#prompt -- in LangChain, you can write your own prompts
#you are prompting LLM to grade and access the documents
system="""You are a grader assessing relevance of a retrieved document to a user question.\n
If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant.\n
Give it a binary score of 'yes' or 'no' score to indicate whether the document is relevant to the question.""" 

#from_message method is used to create a prompt
#takes a list of tuples, each being a specific role and the message
grade_prompt=ChatPromptTemplate.from_messages(
    [
        ("system", system),  #a system message
        ("human", "Retrieved document: \n\n {document} \n\n User Question: {question}") #actual human input and give it a variable using {}
    ]
)

#grade_prompt will be piped into structured_llm_grader
retrieval_grader=grade_prompt | structured_llm_grader

for doc in docs:
    doc_txt=doc.page_content

    #in the invoke, "question" refers to question in the grade_prompt and question is the variable set above about few-shot learning
    #"document" refers to the variable in grade_prompt and will refer to doc_txt
    #we invoke the retrieval_grader with a question and a document and gets automatically assigned by name in grade_prompt
    print(retrieval_grader.invoke({"question":question, "document":doc_txt}), doc_txt[:50], '...', doc.metadata['source'])

###### Generate Components

In [ ]:
from langchain import hub
from langchain_core.output_parsers import StrOutputParser

In [ ]:
#Prompt
#hub is a community hub of prompts instead of the one we created above (grade_prompt)
prompt=hub.pull("rlm/rag-prompt")

for message in prompt.messages:
    print(type(message))
    print(message.prompt.template)
    print('-----')

In [ ]:
#LLM
llm=ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

#post processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#Chain
rag_chain=prompt | llm | StrOutputParser()  #pipe it to print out what comes next using StrOutputParser and get the result you wanted

#run
generation=rag_chain.invoke({"context":format_docs(docs), "question":question})
print(generation)

## Question Rewriter

### Not necessary unless doing a corrective RAG
### One of the bigger corrective parts of the actual process

In [ ]:
#a different LLM to show we can use different one
four_oh_mini=ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

#prompt
system="""You a question re-writer that converts an input question to a better version that is optimized \n
for web search.  Look at the input and try to reason about the underlying semantic intent/meaning."""

rewrite_prompt=ChatPromptTemplate.from_messages(
    [
        ("system", system),
        (
            "human", 
            "Here is an initial question: \n\n{question} \n Formulate an improved question."
        )
    ]
)

question_rewriter=rewrite_prompt | four_oh_mini | StrOutputParser()

question, question_rewriter.invoke({"question":question})

## Building the Graph

### In LangGraph, we have an idea of a state
### A python dictionary (TypeDict)
### Key and values are given specific types that we wanted them to be

In [ ]:
from typing import List
from typing_extensions import TypedDict

In [ ]:
class GraphState(TypedDict):
    """
    Represents the state of our graph

    Attributes:
        question: question
        generation: LLM generation
        documents: List of documents
        times_transformed: number of times the question has been rewritten
    """

    question:str
    generation:str
    documents:List[str]
    times_transformed:int


In [ ]:
from langchain.schema import Document

In [ ]:
def retrieve(state):
    print(state)

    """
    Retrieve documents

    Args: 
        state(dict): the current graph state

    returns:
        state(dict): new key added to state, documents, that contains retrieved documents
    """

    print("---RETRIEVE---")

    question=state["question"]

    #retrieval
    documents=retriever.get_relevant_documents(question)

    return {"documents":documents, "question":question, "times_transformed":0}

In [ ]:
def generate(state):
    """
    Generate answer

    Args:
        state(dict): the current state graph

    Returns:
        state(dict): New key added to state, generation, that contains the llm generation
    """

    print("---GENERATE---")

    question=state["question"]
    documents=state["documents"]

    #RAG generation
    generation=rag_chain.invoke({"context": format_docs(documents), "question":question})
    return {"documents":documents, "question":question, "generation":generation}

In [ ]:

def transform_query(state):
    """
    Transform the query to produce a better question

    Args:
        state(dict): The current graph state

    Returns:
        state(dict): Updates the question key with a re-phrased question
    """

    print("---TRANSFORM QUERY---")

    question=state["question"]
    documents=state["documents"]
    times_transformed=state["times_transformed"]
    times_transformed+=1

    #rewrite question
    better_question=question_rewriter.invoke({"question":question})

    print("---NEW QUESTION---")
    print(better_question)
    
    return {"documents":documents, "question":better_question, "times_transformed":times_transformed}

In [ ]:
def grade_documents(state):
    """
    Determines whether the retrieved documents are relevant to the question

    Args:
        state(dict): The current state graph

    Returns:
        state(dict): updates the keys with only filtered relevant documents
    """

    print("---CHECK DOCUMENTS RELEVANT TO QUESTION---")

    question=state["question"]
    documents=state["documents"]

    #score each doc
    filtered_docs=[]
    web_search="No"

    for d in documents:
        score=retrieval_grader.invoke({"question":question, "document":d.page_content})
        grade=score.binary_score

        print(d.metadata['source'], f'Grade: {grade}')

        if grade == "yes":
            print("---DOCUMENT RELEVANT ---")
            filtered_docs.append(d)

    if len(filtered_docs) == 0:
        print("---GRADE: DOCUMENT NOT RELEVANT---")
        web_search="Yes"

    #event though web_search is not in GraphState it is still allowed
    return {"documents":filtered_docs, "question":question, "web_search":web_search}

In [ ]:
def decide_to_generate(state):
    """
    Determines whether to generate an answer, or re-generate a question

    Args:
        state(dict): current state of the graph

    Returns:
        str: Binary decision for next node to call
    """

    print("---ASSESS GRADED DOCUMENTS---")

    question=state["question"]
    web_search=state["web_search"]

    if web_search=="Yes":
        #check times_transformed
        if state["times_transformed"]>3:
            print("---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO THE QUESTION AND HAVE TRANSFORMED IT 3 TIMES---")
            print("---JUST DO SOMETHING---")

            return "should_generate"
        
        #all documents have been filtered check_relevance
        #we will generate a new query
        print("---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION.  TRANSFORM QUERY---")

        return "should_transform_query"
    else:
        #we have relevant documents
        print("---DECISION: GENERATE---")

        return "should_generate"


In [ ]:
from langgraph.graph import END, StateGraph, START

workflow=StateGraph(GraphState)

#define the node
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)

#build the graph
workflow.add_edge(START, "retrieve")  #first thing that we're going to do is retrieve
workflow.add_edge("retrieve", "grade_documents") #from retrieve we go to grade_documents
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,   #calls this method to decide to generate return either should_transform_query or should_generate
    {
        "should_transform_query":"transform_query",  #if method response with should_transform_query, it should go to transform_query
        "should_generate":"generate",                #if method returns with should_generate go to generate
    },
)

workflow.add_edge("transform_query", "grade_documents")  #if we in transform_query go back to grade_documents
workflow.add_edge("generate", END)

app=workflow.compile()


In [ ]:
#run
inputs={"question": "What on earth is few-shots learning?"}

for output in app.stream(inputs):
    for key, value in output.items():
        #Node
        print(f"Node '{key}':")

print(value["generation"])

In [ ]:
#run
inputs={"question": "How big is the moon?"}

for output in app.stream(inputs):
    for key, value in output.items():
        #Node
        print(f"Node '{key}':")

print(value["generation"])

In [1]:
#visualize the graph
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    pass